In [16]:
# Import library satu kata
import os
import pandas as pd
import pickle
import string
import spacy

# Import library NLTK
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist

# Import library sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load PATH
MODEL_PATH = "./NB_model.pkl"
CSV_PATH = "./jobpostingdata.csv"

: 

In [ ]:
lemmatizer = WordNetLemmatizer()
eng_stopwords = stopwords.words("english")
punctuation = string.punctuation

: 

In [ ]:
# Preprocessing: get_tag(tag), lematizing(word_list), preprocessing(sentence)
def get_tag(tag):
    if tag == 'jj':
        return 'a'
    elif tag in ['rb','vb','nn']:
        return tag[0]
    else:
        return 'n'
    
def lemmatizing(word_list):
    lemma_list = []
    tagged = pos_tag(word_list)
    for word, label in tagged:
        tag = get_tag(label.lower())
        if label:
            result = lemmatizer.lemmatize(word, tag)
            lemma_list.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemma_list.append(result)
    return lemma_list

def preprocessing(sentence):
    tokenization = word_tokenize(sentence.lower())
    removed_stopwords = [token for token in tokenization if token not in eng_stopwords]
    removed_punctuation = [token for token in removed_stopwords if token not in punctuation]
    lemma_list = lemmatizing(removed_punctuation)
    return lemma_list

: 

In [ ]:
sentence = "Today is a good day for swimming"
print(f"Before: {sentence}\nAfter: {preprocessing(sentence)}")

Before: Today is a good day for swimming
After: ['today', 'good', 'day', 'swimming']


: 

In [ ]:
# Load dataset
df = pd.read_csv(CSV_PATH)
df = df.drop_duplicates(subset='title')

X = df['text']
y = df['fraudulent'].map({0: "Real Job Posting", 1: "Fake Job posting"})

all_sentence = " ".join(X)
all_preprocessed = preprocessing(all_sentence)
freqdist = FreqDist(all_preprocessed)

print(freqdist.most_common(10))

[('work', 971), ('experience', 894), ('team', 715), ('’', 685), ('service', 684), ('company', 672), ('customer', 665), ('product', 613), ('business', 503), ('skill', 480)]


: 

In [ ]:
# Feature Extraction
def feature_extraction(text):
    features = {}
    for word in freqdist.keys():
        features[word] = (word in text)
    return features

features_set = [(feature_extraction(preprocessing(text)), fraudulent) for (text, fraudulent) in zip(X,y)]
print(features_set[0])

({'php': True, 'developer': True, "'re": True, 'skilled': True, 'build': True, 'release': True, 'improve': True, 'grow': True, 'every': True, 'day': True, 'numerous': True, 'challenge': True, 'take': True, 'next': True, 'step': True, 'know': True, "'s": True, 'important': True, 'time': True, 'get': True, 'person': True, 'match': True, 'top': True, 'notch': True, 'company': True, 'fit': True, 'like': True, 'glove': True, 'kind': True, 'potential': True, 'you.to': True, 'make': True, 'sure': True, 'made': True, 'heaven': True, 'talk': True, 'first': True, 'test': True, 'skill': True, 'level': True, 'see': True, 'want': True, 'career': True, 'particular.when': True, 'introduce': True, 'match.if': True, 'question': True, 'sound': True, 'familiar': True, 'answer': True, 'introducing': True, 'doingcoming': True, 'new': True, 'functionality': True, 'building': True, 'them.improving': True, 'technology': True, 'run': True, '300': True, 'web': True, 'shops.improving': True, 'user-friendliness':

: 

In [ ]:
# Train & test set
import random
random.shuffle(features_set)

train_count = int(len(features_set) * 0.8)
train_set = features_set[:train_count]
test_set = features_set[train_count:]

: 

In [ ]:
# Train model
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    acc = accuracy(classifier, test_set)
    print(f"Accuracy: {acc *100}%")

    with open(MODEL_PATH, "wb") as f:
        pickle.dump(classifier, f)
    return classifier


: 

In [ ]:
def analyze_statement(classifier, sentence):
    preprocessed_txt = preprocessing(sentence)
    extracted_txt = feature_extraction(preprocessed_txt)
    prediction = classifier.classify(extracted_txt)
    return prediction

: 

In [ ]:
def write_statement():
    while True:
        statement = input("Enter statement: ")
        if len(statement) < 20:
            print("Statement must contain more than 20 character")
        elif len(statement.split()) < 3:
            print("Statement must contain more than 3 word")
        else:
            return statement

: 

In [17]:
# Insialisasi 3 hal
current_txt = None
categories_txt = None
classifier = None

while True:
    print(f"\n{'=' * 60} RIL or FEK {'=' * 60}")
    print("Job Posting Classification\nReal/Fake job posting classification & job reccomendation based on text")
    print(f"Your text: {current_txt}")
    print(f"Your text categories: {categories_txt}")
    print("1. Write your text\n2. View job posting recommendations based on text\n3. View NER\n4. Exit")

    choice = input("Enter choice: ")

    if choice == "1":
        current_txt = write_statement()
        if os.path.exists(MODEL_PATH):
            print("Load model...")
            with open(MODEL_PATH, "rb") as f:
                classifier = pickle.load(f)
        else:
            classifier = train_model()
        categories_txt = analyze_statement(classifier, current_txt)

    elif choice == "2":
        # Validate that current_txt is not None
        if current_txt == None:
            print("No text\nEnter to continue")
            input("(click enter)")
            continue

        # Initialize vectorize and tfidfmatrix
        vectorizer = TfidfVectorizer(stop_words='english')
        corpus = df['text']
        tfidf_matrix = vectorizer.fit_transform(corpus)

        # Find user_vec, similarity_score, & top_jobs
        user_vec = vectorizer.transform([" ".join(word_tokenize(current_txt.lower()))])
        similarity_Score = cosine_similarity(user_vec, tfidf_matrix).flatten() # Mengubah dari 2D ke 1D
        top_jobs = similarity_Score.argsort()[-6:-1][::-1]

        # Display the top jobs using for loops and enumerate
        for i, idx in enumerate(top_jobs, 1):
            print(f"{i}. {df.iloc[idx]['title']}")
        
        input("(enter to continue)")

    elif choice == "3":
        # Validate if current_txt is not None
        if current_txt == None:
            print("No text\nEnter to continue")
            input("(click enter)")
            continue

        # Initialize nlp, doc, & categories
        nlp = spacy.load("en_core_web_sm")
        doc = nlp(current_txt)
        categories = {}

        # find all the labels and assigne each text to it's proper label
        for ent in doc.ents:
            label = ent.label_
            if label not in categories:
                categories[label] = []
            categories[label].append(ent.text)

        # Display NER
        print("\nNER")
        for label, entities in categories.items():
            print(f"{label}: {' ,'.join(entities)}")

        input("(enter to continue)")

    elif choice == "4":
        print("Thank you")
        break
    else:
        print("Invalid option")


============================================================ RIL or FEK ============================================================
Job Posting Classification
Real/Fake job posting classification & job reccomendation based on text
Your text: None
Your text categories: None
1. Write your text
2. View job posting recommendations based on text
3. View NER
4. Exit
Load model...

============================================================ RIL or FEK ============================================================
Job Posting Classification
Real/Fake job posting classification & job reccomendation based on text
Your text: Management System Administrator Aker Solutions is a global provider of products, systems and services to the oil and gas industry. Our engineering, design and technology bring discoveries into production and maximize recovery from each petroleum field. We employ approximately 28,000 people in about 30 countries. Go to #URL_0fa3f7c5e23a16de16a841e368006cae916884407d90b154dfef

: 